# 24 — Order Book / Matching Engine (Amazon FAR style)

A limit order book with price-time priority matching (the core of an exchange). Parts 1-3 build the
core (concurrency at Part 3); Parts 4-5 are stretch tasks (cancel + validation, then parallel
per-symbol replay). Fill stubs, run each test cell, peek at the collapsed `ref_` solutions only after
trying.

Bids (buys) want the **highest** price; asks (sells) want the **lowest**. Within a price level, orders
fill **first-in-first-out**.

---

## Part 1 — Resting book

`OrderBook` with `add_limit(side, price, qty, oid)` (rests an order), `best_bid()` / `best_ask()`
(`None` if empty), and `depth(side, price)` (total resting qty at a level).

**Lock down:** bids sorted high→low, asks low→high; FIFO within a level; "best" bid is the max price,
best ask the min.

In [ ]:
from collections import deque, defaultdict


class OrderBook:
    def __init__(self):
        raise NotImplementedError

    def add_limit(self, side, price, qty, oid):
        raise NotImplementedError

    def best_bid(self):
        raise NotImplementedError

    def best_ask(self):
        raise NotImplementedError

    def depth(self, side, price):
        raise NotImplementedError

In [ ]:
# --- Part 1 tests ---
def _t1():
    b = OrderBook()
    b.add_limit("buy", 100, 5, "o1"); b.add_limit("buy", 101, 3, "o2"); b.add_limit("sell", 105, 2, "o3")
    assert b.best_bid() == 101 and b.best_ask() == 105
    assert b.depth("buy", 100) == 5
    b.add_limit("buy", 100, 2, "o4")
    assert b.depth("buy", 100) == 7
    print("Part 1 OK")

_t1()

## Part 2 — Matching

`match(book, side, qty) -> list[(price, fill_qty)]`: a market order of `side` consumes the opposite
side by **price-time priority** (best price first, FIFO within a level) until `qty` is filled or the
book runs out. It mutates the book (removing/decrementing consumed orders) and returns the fills.

**Lock down:** a buy lifts the lowest asks, a sell hits the highest bids; partial fills decrement the
resting order; an exhausted level is removed.

In [ ]:
def match(book, side, qty):
    raise NotImplementedError

In [ ]:
# --- Part 2 tests ---
def _t2():
    b = OrderBook()
    b.add_limit("sell", 105, 2, "a"); b.add_limit("sell", 105, 3, "b"); b.add_limit("sell", 106, 10, "c")
    assert match(b, "buy", 4) == [(105, 2), (105, 2)]   # a fully, b partially (time priority)
    assert b.depth("sell", 105) == 1
    assert match(b, "buy", 10) == [(105, 1), (106, 9)]  # rest of b, then 106
    assert b.depth("sell", 106) == 1
    print("Part 2 OK")

_t2()

## Part 3 — Thread-safe book

`ConcurrentOrderBook`: thread-safe `add_limit`, `match`, `depth`. Many threads submit and match at
once with no double-fills or lost orders.

**The invariant:** with 100 units resting on the ask side and 200 threads each `match("buy", 1)`,
**exactly 100 units** fill (no over-fill, no resting order consumed twice). **Discuss:** the whole
add/match must be atomic (check-then-consume race); per-symbol locking; lock-free matching is hard.

In [ ]:
import threading


class ConcurrentOrderBook:
    def __init__(self):
        raise NotImplementedError

    def add_limit(self, side, price, qty, oid):
        raise NotImplementedError

    def match(self, side, qty):
        raise NotImplementedError

    def depth(self, side, price):
        raise NotImplementedError

In [ ]:
# --- Part 3 tests ---
def _t3():
    b = ConcurrentOrderBook()
    for i in range(100):
        b.add_limit("sell", 105, 1, "a%d" % i)
    filled, lock = [], threading.Lock()

    def worker():
        f = b.match("buy", 1)
        with lock:
            filled.append(sum(q for _, q in f))

    ts = [threading.Thread(target=worker) for _ in range(200)]   # 200 buyers, 100 units
    for t in ts: t.start()
    for t in ts: t.join()
    assert sum(filled) == 100                            # exactly the available units, no double-fill
    print("Part 3 OK")

_t3()

## Part 4 (stretch) — Cancel & validation

`CancelableOrderBook`: `add_limit(side, price, qty, oid)` tracks each order by id; `cancel(oid) ->
bool` removes a resting order (False if unknown); `depth(side, price)`. Reject bad input: `qty <= 0`,
unknown `side`, or a duplicate `oid` raise `ValueError`.

**Discuss:** an id→location index for O(1) cancel lookup, self-trade prevention, and validating before
mutating so a rejected order leaves the book untouched.

In [ ]:
class CancelableOrderBook:
    def __init__(self):
        raise NotImplementedError

    def add_limit(self, side, price, qty, oid):
        raise NotImplementedError

    def cancel(self, oid):
        raise NotImplementedError

    def depth(self, side, price):
        raise NotImplementedError

In [ ]:
# --- Part 4 tests ---
def _t4():
    b = CancelableOrderBook()
    b.add_limit("buy", 100, 5, "o1"); b.add_limit("buy", 100, 3, "o2")
    assert b.depth("buy", 100) == 8
    assert b.cancel("o1") is True and b.depth("buy", 100) == 3
    assert b.cancel("nope") is False
    for bad in (0, -1):
        try:
            b.add_limit("buy", 100, bad, "x")
        except ValueError:
            pass
        else:
            raise AssertionError("expected ValueError for qty=%r" % bad)
    try:
        b.add_limit("weird", 100, 1, "y")
    except ValueError:
        pass
    else:
        raise AssertionError("expected ValueError for bad side")
    print("Part 4 OK")

_t4()

## Part 5 (stretch) — Parallel per-symbol replay

`parallel_replay(books, num_procs) -> dict[symbol, traded_volume]`: each symbol's order log is replayed
through its own book independently, across processes with `ProcessPoolExecutor`; worker is
`orderbook_workers.replay_orders`. `books` is `dict[symbol, list[order]]` where an order is
`("limit", side, price, qty, oid)` or `("market", side, qty)`.

**Discuss:** different symbols are independent books (embarrassingly parallel by symbol); within a
symbol order matters (sequential); sharding the exchange by symbol.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
import orderbook_workers


def parallel_replay(books, num_procs) -> dict:
    raise NotImplementedError

In [ ]:
# --- Part 5 tests ---
def _t5():
    books = {
        "AAPL": [("limit", "sell", 105, 5, "a"), ("limit", "sell", 106, 5, "b"), ("market", "buy", 7)],
        "GOOG": [("limit", "buy", 50, 3, "c"), ("market", "sell", 10)],
    }
    assert parallel_replay(books, 2) == {"AAPL": 7, "GOOG": 3}
    print("Part 5 OK")

_t5()

## Likely follow-ups
- Limit orders that cross the book should match immediately, then rest the remainder.
- Order types: IOC / FOK / stop / iceberg; self-trade prevention; fee/priority models.
- Market-data feed (L2 book snapshots + deltas); sequencer / single-writer design for determinism.

---
## Reference solutions (don't peek until you've tried)

In [ ]:
from collections import deque, defaultdict
import threading
from concurrent.futures import ProcessPoolExecutor
import orderbook_workers


class RefOrderBook:
    def __init__(self):
        self.bids = defaultdict(deque)
        self.asks = defaultdict(deque)

    def add_limit(self, side, price, qty, oid):
        (self.bids if side == "buy" else self.asks)[price].append((oid, qty))

    def best_bid(self):
        ps = [p for p, d in self.bids.items() if d]
        return max(ps) if ps else None

    def best_ask(self):
        ps = [p for p, d in self.asks.items() if d]
        return min(ps) if ps else None

    def depth(self, side, price):
        book = self.bids if side == "buy" else self.asks
        return sum(q for _, q in book.get(price, ()))


def _match(bids, asks, side, qty):
    fills = []
    levels = asks if side == "buy" else bids
    pick = min if side == "buy" else max
    rem = qty
    while rem > 0:
        prices = [p for p, d in levels.items() if d]
        if not prices:
            break
        best = pick(prices)
        q = levels[best]
        while rem > 0 and q:
            oid, oqty = q[0]
            take = min(rem, oqty)
            fills.append((best, take))
            rem -= take
            if take == oqty:
                q.popleft()
            else:
                q[0] = (oid, oqty - take)
        if not q:
            del levels[best]
    return fills


def ref_match(book, side, qty):
    return _match(book.bids, book.asks, side, qty)


class RefConcurrentOrderBook:
    def __init__(self):
        self.bids = defaultdict(deque)
        self.asks = defaultdict(deque)
        self.lock = threading.Lock()

    def add_limit(self, side, price, qty, oid):
        with self.lock:
            (self.bids if side == "buy" else self.asks)[price].append((oid, qty))

    def match(self, side, qty):
        with self.lock:
            return _match(self.bids, self.asks, side, qty)

    def depth(self, side, price):
        with self.lock:
            book = self.bids if side == "buy" else self.asks
            return sum(q for _, q in book.get(price, ()))


class RefCancelableOrderBook:
    def __init__(self):
        self.bids = defaultdict(deque)
        self.asks = defaultdict(deque)
        self.loc = {}

    def add_limit(self, side, price, qty, oid):
        if qty <= 0:
            raise ValueError("qty must be positive")
        if side not in ("buy", "sell"):
            raise ValueError("bad side")
        if oid in self.loc:
            raise ValueError("duplicate oid")
        (self.bids if side == "buy" else self.asks)[price].append((oid, qty))
        self.loc[oid] = (side, price)

    def cancel(self, oid):
        if oid not in self.loc:
            return False
        side, price = self.loc.pop(oid)
        q = (self.bids if side == "buy" else self.asks)[price]
        for i, (o, _) in enumerate(q):
            if o == oid:
                del q[i]
                break
        return True

    def depth(self, side, price):
        book = self.bids if side == "buy" else self.asks
        return sum(q for _, q in book.get(price, ()))


def ref_parallel_replay(books, num_procs):
    with ProcessPoolExecutor(max_workers=num_procs) as ex:
        return dict(ex.map(orderbook_workers.replay_orders, list(books.items())))


_b = RefOrderBook(); _b.add_limit("buy", 10, 5, "x"); _b.add_limit("sell", 12, 4, "y")
assert _b.best_bid() == 10 and _b.best_ask() == 12
assert ref_match(_b, "buy", 3) == [(12, 3)] and _b.depth("sell", 12) == 1
_c = RefCancelableOrderBook(); _c.add_limit("buy", 5, 2, "o1")
assert _c.cancel("o1") and not _c.cancel("o1")
assert ref_parallel_replay({"S": [("limit", "sell", 1, 5, "a"), ("market", "buy", 3)]}, 2) == {"S": 3}
print("reference solutions OK")